# OD-QC-001 — Ekpyrotic contraction, anisotropy smoothing, and WKB classicality

**Research question.** Reproduce the smoothing behaviour of an ekpyrotic
contracting universe, then examine the additional assumptions needed to
describe a bounce and semiclassical evolution.

**Starting relationship.** With a scalar-field energy component scaling as
$\rho_\phi \propto a^{-3(1+w)}$ and a shear (anisotropy) component scaling
as $\rho_{\rm shear} \propto a^{-6}$,

$$\frac{\rho_{\rm shear}}{\rho_\phi} \propto a^{\,3(w-1)},$$

so for a contracting universe ($a$ decreasing) the anisotropy loses relative
to the scalar field exactly when $w > 1$.

**Model (reduced Planck units, $M_{\rm Pl}=1$, so $3H^2 = \rho$).** Flat
FLRW background plus a canonical scalar field in the ekpyrotic exponential
potential

$$V(\phi) = -V_0\,e^{-c\phi},\qquad c = \sqrt{2/p},\qquad 0 < p \ll 1,$$

whose scaling attractor is $a \propto |t|^{p}$ ($t<0$) with
$w = -1 + \tfrac{2}{3p} > 1$. Anisotropy is evolved as a shear energy
density $\rho_\Sigma \equiv R$ obeying the exact Bianchi I relation
$\dot R = -6 H R$.

**Experiment progression** (as specified in the task): (1) contraction for
different equations of state; (2) anisotropy evolution and numerical
convergence; (3) contracting GR solution vs a separately specified effective
bounce model; (4) the WKB classicality condition (Lehners 2015 context);
(5) documentation of predictions, assumptions, and open items. Steps are the
sections of this notebook; step 5 is expanded in
`results/OD-QC-001_REPORT.md`.


In [1]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import ekpyrotic as ek

FIGS = ROOT / 'results' / 'figs'
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

metrics = {'steps': {}}

# Global parameters
V0 = 1.0
t_i = -1.0e6          # start of the ekpyrotic phase (a(t_i) = 1)
SHEAR_FRAC = 1.0e-4   # rho_Sigma / rho_phi at t_i
RHO_STOP = 1.0e10     # trust cutoff for the classical GR integration
P_LIST = [0.01, 0.05, 0.1, 0.3, 0.5]

print('p        w        3(w-1)   n_e-folds to rho_stop')
for p in P_LIST:
    rho_i = 3 * (p / t_i) ** 2
    n_f = 0.5 * p * np.log(RHO_STOP / rho_i)
    print(f'{p:<8} {ek.w_of_p(p):<8.3f} {ek.shear_ratio_exponent(p):<8.2f} {n_f:.2f}')


p        w        3(w-1)   n_e-folds to rho_stop
0.01     65.667   194.00   0.29
0.05     12.333   34.00    1.39
0.1      5.667    14.00    2.71
0.3      1.222    0.67     7.79
0.5      0.333    -2.00    12.74


## Step 1 — Classical contraction for different equations of state

For each $p$ we place the initial data exactly on the scaling attractor,
add a small shear contamination ($\rho_\Sigma/\rho_\phi = 10^{-4}$), and
integrate the GR system in contraction e-folds $n = -\ln a$ until the total
density reaches the cutoff $\rho_{\rm stop} = 10^{10}$ (a numerical trust
cutoff; classically the crunch is at finite time).

Checks:

* $\rho_\phi \propto a^{-3(1+w)}$ with $w = -1 + 2/(3p)$;
* $\rho_\Sigma/\rho_\phi \propto a^{3(w-1)}$ — fitted exponent vs theory;
* the $p = 0.5$ case ($w = 1/3 < 1$) acts as the counterexample: the ratio
  grows during contraction, i.e. no smoothing.

All exponents are fitted in the window where $\rho_\Sigma/\rho_\phi < 10^{-3}$,
i.e. while the shear is still a perturbation on the scalar-field-driven
background. For $w < 1$ the ratio leaves this window: the shear backreacts
and takes over the contraction — that takeover *is* the failure mode, made
quantitative here.

**Assumptions in play:** flat slicing, single canonical field, exact
exponential potential, shear treated as an adiabatic stiff component
(exact for Bianchi I with isotropic source), no backreaction of
inhomogeneous modes.


In [2]:
runs = {}
rows = []
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True)

for p in P_LIST:
    sol, pot = ek.integrate_gr(p, V0, t_i, SHEAR_FRAC, rho_stop=RHO_STOP)
    n = np.linspace(0.0, sol.t[-1], 2000)
    y = sol.sol(n)
    phi, u, h, R, t = y
    a = np.exp(-n)
    rho_p = 0.5 * u**2 + pot[0](phi)
    ratio = R / rho_p
    runs[p] = dict(sol=sol, pot=pot, n=n, y=y, a=a, rho_p=rho_p, ratio=ratio)

    # fit in the window where the shear is still a perturbation
    # (for w < 1 the shear grows and eventually backreacts; that *is* the
    # failure of smoothing, but the clean exponent is measured earlier)
    win = ratio < 1e-3
    if win.sum() > 30:
        m_r, r2_r = ek.fit_exponent(a[win], ratio[win])
        m_p, r2_p = ek.fit_exponent(a[win], rho_p[win])
    else:
        m_r, r2_r = ek.fit_exponent(a, ratio)
        m_p, r2_p = ek.fit_exponent(a, rho_p)
    rows.append((p, ek.w_of_p(p), m_p, -3*(1+ek.w_of_p(p)),
                 m_r, ek.shear_ratio_exponent(p), r2_r))

    ax1.loglog(a, rho_p, label=f'p={p}')
    ax2.loglog(a, ratio, label=f'p={p}')

# reference slopes (through the last point of the p=0.1 curve)
p_ref = 0.1
r0 = runs[p_ref]
a_ref = np.logspace(np.log10(r0['a'].min()), 0, 50)
ax1.loglog(a_ref, r0['rho_p'][-1]*(a_ref/r0['a'][-1])**(-2/p_ref),
           'k--', lw=1, label=r'$\propto a^{-2/p}$')
ax2.loglog(a_ref, r0['ratio'][-1]*(a_ref/r0['a'][-1])**ek.shear_ratio_exponent(p_ref),
           'k--', lw=1, label=r'$\propto a^{3(w-1)}$')

for ax, ylab in ((ax1, r'$\rho_\phi$'), (ax2, r'$\rho_\Sigma/\rho_\phi$')):
    ax.set_xlabel('scale factor a (contraction: decreasing)')
    ax.set_ylabel(ylab)
    ax.invert_xaxis()
    ax.legend(fontsize=8)
ax1.set_title('Scalar-field density vs a')
ax2.set_title('Anisotropy-to-field ratio vs a')
fig.tight_layout()
fig.savefig(FIGS / 'fig1_smoothing.png')
plt.show()

print('p      w      rho_phi exp (fit/theory)      ratio exp (fit/theory)   R^2')
for r in rows:
    print(f'{r[0]:<6} {r[1]:<7.3f} {r[2]:>10.4f} / {r[3]:>9.4f}   '
          f'{r[4]:>10.4f} / {r[5]:>9.4f}   {r[6]:.10f}')

metrics['steps']['1'] = {
    'table': [{'p': r[0], 'w': r[1], 'rho_phi_exp_fit': r[2],
               'rho_phi_exp_theory': r[3], 'ratio_exp_fit': r[4],
               'ratio_exp_theory': r[5], 'ratio_fit_R2': r[6]} for r in rows],
}


p      w      rho_phi exp (fit/theory)      ratio exp (fit/theory)   R^2
0.01   65.667   -199.9999 / -200.0000     193.9999 /  194.0000   1.0000000000
0.05   12.333    -40.0000 /  -40.0000      34.0000 /   34.0000   1.0000000000
0.1    5.667     -20.0000 /  -20.0000      14.0000 /   14.0000   1.0000000000
0.3    1.222      -6.6665 /   -6.6667       0.6665 /    0.6667   0.9999999993
0.5    0.333      -4.0007 /   -4.0000      -1.9993 /   -2.0000   0.9999999758


## Step 2 — Anisotropy evolution: sensitivity and numerical convergence

Three checks:

1. **Tolerance sweep.** The $p=0.1$ run at four tolerance levels
   (DOP853). We report the maximum relative Friedmann-constraint residual
   (an independent quantity: $\mathcal C = 3h^2 - \rho_{\rm tot}$ is not
   enforced by the evolution), the self-convergence of $u(n)$ against the
   finest run, and the stability of the fitted exponent.
2. **Shear-amplitude sensitivity.** Initial fractions
   $10^{-6}, 10^{-4}, 10^{-2}$: the exponent must be unchanged and the final
   ratio must scale linearly with the initial fraction.
3. **$V_0$ sensitivity.** The exponent must be independent of $V_0$
   ($V_0$ shifts $\phi_0$ only).

*Note on conditioning:* the time-translation mode of the scaling solution is
neutral — errors in the cosmic-time coordinate $t(n)$ grow as $e^{n/p}$.
All physical statements below are therefore made in $n$ (or $a$), which is
well conditioned; $t$ is only used where its accuracy is adequate.


In [3]:
p = 0.1
tols = [(1e-7, 1e-9), (1e-9, 1e-11), (1e-11, 1e-13), (1e-13, 1e-15)]
sweep = []
finest = None
for rtol, atol in tols:
    sol, pot = ek.integrate_gr(p, V0, t_i, SHEAR_FRAC, rho_stop=RHO_STOP,
                               rtol=rtol, atol=atol)
    n = np.linspace(0.0, sol.t[-1], 1500)
    y = sol.sol(n)
    C = ek.constraint_residual_gr(pot, y)
    rho = ek.rho_tot(pot, y[0], y[1], y[3])
    a = np.exp(-n)
    rho_p = 0.5 * y[1]**2 + pot[0](y[0])
    m, _ = ek.fit_exponent(a, y[3] / rho_p)
    rec = dict(rtol=rtol, atol=atol, n=n, y=y,
               C_rel_max=float(np.max(np.abs(C) / rho)),
               m=float(m))
    if finest is not None:
        u_ref = finest['sol'].sol(n)[1]
        rec['u_self_diff'] = float(np.max(np.abs(y[1] - u_ref) / np.abs(u_ref)))
    else:
        rec['sol'] = sol
        rec['u_self_diff'] = np.nan
    rec['pot'] = pot
    sweep.append(rec)
finest = sweep[-1]
for i, rec in enumerate(sweep[:-1]):
    u_ref = finest['sol'].sol(rec['n'])[1]
    rec['u_self_diff'] = float(np.max(np.abs(rec['y'][1] - u_ref) / np.abs(u_ref)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
for rec in sweep:
    C = np.abs(ek.constraint_residual_gr(rec['pot'], rec['y']))
    rho = ek.rho_tot(rec['pot'], rec['y'][0], rec['y'][1], rec['y'][3])
    ax1.semilogy(rec['n'], C / rho, label=f"rtol={rec['rtol']:.0e}")
ax1.set_xlabel('e-folds n'); ax1.set_ylabel(r'$|\mathcal{C}|/\rho_{\rm tot}$')
ax1.set_title('Constraint residual along the run'); ax1.legend(fontsize=8)

ax2.loglog([r['rtol'] for r in sweep[:-1]], [r['u_self_diff'] for r in sweep[:-1]],
           'o-', label='max rel diff of u(n) vs finest run')
ax2.set_xlabel('rtol'); ax2.set_ylabel('self-convergence of u')
ax2.set_title('Self-convergence'); ax2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / 'fig2_convergence.png')
plt.show()

print('Convergence sweep (p = 0.1)')
print('rtol        max |C|/rho    exponent m (theory 14)   u self-diff')
for r in sweep:
    print(f"{r['rtol']:<11.0e} {r['C_rel_max']:<14.3e} {r['m']:<24.6f} {r['u_self_diff']:.3e}")

# shear amplitude sensitivity
print()
print('Shear-amplitude sensitivity (p = 0.1)')
print('shear_frac   exponent m      ratio at cutoff')
shear_rows = []
for sf in (1e-6, 1e-4, 1e-2):
    sol, pot = ek.integrate_gr(p, V0, t_i, sf, rho_stop=RHO_STOP)
    n = np.linspace(0, sol.t[-1], 500)
    y = sol.sol(n)
    a = np.exp(-n)
    rho_p = 0.5 * y[1]**2 + pot[0](y[0])
    ratio = y[3] / rho_p
    m, _ = ek.fit_exponent(a, ratio)
    shear_rows.append({'shear_frac': sf, 'm': float(m),
                       'ratio_final': float(ratio[-1])})
    print(f'{sf:<12.0e} {m:<15.6f} {ratio[-1]:.3e}')

# V0 sensitivity
print()
print('V0 sensitivity (p = 0.1): exponents ', end='')
v0_rows = []
for v0 in (0.1, 1.0, 10.0):
    sol, pot = ek.integrate_gr(p, v0, t_i, SHEAR_FRAC, rho_stop=RHO_STOP)
    n = np.linspace(0, sol.t[-1], 500)
    y = sol.sol(n)
    a = np.exp(-n)
    rho_p = 0.5 * y[1]**2 + pot[0](y[0])
    m, _ = ek.fit_exponent(a, y[3] / rho_p)
    v0_rows.append({'V0': v0, 'm': float(m)})
    print(f'V0={v0}: {m:.6f}   ', end='')
print()

metrics['steps']['2'] = {
    'p': p,
    'tolerance_sweep': [{k: v for k, v in r.items()
                         if k in ('rtol', 'C_rel_max', 'm', 'u_self_diff')}
                        for r in sweep],
    'shear_sensitivity': shear_rows,
    'V0_sensitivity': v0_rows,
}


Convergence sweep (p = 0.1)
rtol        max |C|/rho    exponent m (theory 14)   u self-diff
1e-07       3.192e-06      13.999979                1.898e-06
1e-09       1.971e-08      13.999978                1.266e-08
1e-11       1.891e-10      13.999978                1.035e-10
1e-13       1.738e-12      13.999978                nan

Shear-amplitude sensitivity (p = 0.1)
shear_frac   exponent m      ratio at cutoff
1e-06        14.000000       3.420e-23
1e-04        13.999978       3.420e-21
1e-02        13.997786       3.434e-19

V0 sensitivity (p = 0.1): exponents V0=0.1: 13.999978   V0=1.0: 13.999978   V0=10.0: 13.999978   


## Step 3 — Contracting GR solution vs a separately specified effective bounce

The GR solution classically terminates: $\rho \to \infty$ at finite time
(the cutoff $\rho_{\rm stop}$ above only marks where we stop trusting it).
No bounce exists inside this model.

To study what a bounce would *add*, we specify, by hand, the effective
Friedmann equation used in loop-quantum-cosmology-style phenomenology,

$$H^2 = \frac{\rho}{3}\Big(1 - \frac{\rho}{\rho_c}\Big),$$

together with the unchanged continuity equation, which fixes the modified
Raychaudhuri equation $\dot H = -\tfrac12(\rho+p)\big(1 - 2\rho/\rho_c\big)$.
Critical density $\rho_c = 0.41\,\rho_{\rm Pl}$ (the standard effective
value) with a $\pm 1$ decade sensitivity sweep.

This dynamics (i) bounces at $\rho = \rho_c$ where the GR solution
approaches the crunch, and (ii) — with our negative potential — **recollapses
after the bounce**: in expansion the kinetic energy redshifts as $a^{-6}$
until $\rho = 0$ again. The run therefore covers exactly one
contraction–bounce–expansion window; joining the expanding branch onto a
radiation-dominated hot big bang requires physics that this specified model
does not contain (a potential that turns positive, reheating, etc.).


In [4]:
p = 0.1
RHO_C_LIST = [0.041, 0.41, 4.1]

# GR reference integrated up to rho = max(rho_c) (the largest bounce density)
sol_gr, pot_gr = ek.integrate_gr(p, V0, t_i, SHEAR_FRAC, rho_stop=max(RHO_C_LIST))
ngr = np.linspace(0, sol_gr.t[-1], 3000)
ygr = sol_gr.sol(ngr)
tgr = ygr[4]
agr = np.exp(-ngr)
rho_gr = ek.rho_tot(pot_gr, ygr[0], ygr[1], ygr[3])
H_gr = -ygr[2]

eff_runs = {}
bounce_table = []
for rho_c in RHO_C_LIST:
    sole, pote = ek.integrate_effective(p, V0, t_i, SHEAR_FRAC, rho_c=rho_c)
    tb = float(sole.t_events[0][0])
    yb = sole.sol(tb)
    rho_b = ek.rho_tot(pote, yb[0], yb[1], yb[3])
    a_b = float(np.exp(yb[4]))
    # GR scale factor at the same density: a ∝ rho^{-p/2} on the attractor
    n_at_rhoc = 0.5 * p * np.log(rho_c / (3 * (p / t_i) ** 2))
    a_gr_at_rhoc = float(np.exp(-n_at_rhoc))
    bounce_table.append({'rho_c': rho_c, 't_bounce': tb,
                         'a_bounce': a_b, 'rho_at_bounce': float(rho_b),
                         'a_GR_at_rho_c': a_gr_at_rhoc})
    eff_runs[rho_c] = dict(sol=sole, pot=pote, tb=tb)
    print(f'rho_c={rho_c:<6}  t_bounce={tb:>10.4f}  a_bounce={a_b:.4f}  '
          f'rho(t_bounce)={rho_b:.6f}  a_GR(rho=rho_c)={a_gr_at_rhoc:.4f}')

# smoothing achieved by the time the GR solution reaches rho_c = 0.41
r01 = runs[p]
n_sm = 0.5 * p * np.log(0.41 / (3 * (p / t_i) ** 2))
smooth_factor = float(np.exp(-(2 / p - 6) * n_sm))
print(f'\nBy rho = 0.41 the contraction lasted {n_sm:.2f} e-folds and the '
      f'anisotropy ratio fell by a factor {smooth_factor:.2e} (p=0.1).')

# Reference crunch time t_c of the GR solution: the time-translation mode of
# the ekpyrotic attractor makes the absolute location of the crunch in t
# extremely sensitive to any homogeneous perturbation (here: the shear's
# backreaction), so we measure t_c from the trajectory itself (late-time
# plateau of t - p/H) and plot everything in crunch-centred time t - t_c.
ygr_late = ygr[:, -50:]
t_c = float(np.mean(ygr_late[4] + p / -ygr_late[2]))  # t - p/H, H = -h
tgr_s = tgr - t_c
print(f'GR crunch time offset t_c = {t_c:.4f} (time-translation mode; '
      f'linear in the initial shear fraction)')

# post-bounce expansion factor before recollapse (rho_c = 0.41 run)
r041 = eff_runs[0.41]
ts_post = np.linspace(r041['tb'], r041['sol'].t[-1], 3000)
a_post = np.exp(r041['sol'].sol(ts_post)[4])
exp_factor = float(a_post.max() / a_post[0])
print(f'Post-bounce expansion before recollapse: a_max/a_bounce = '
      f'{exp_factor:.4f}')

fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))

ax = axs[0]
ax.semilogx(-tgr_s, agr, 'k-', lw=2, label='GR (ends in crunch)')
for rho_c in RHO_C_LIST:
    r = eff_runs[rho_c]
    ts = np.linspace(t_i, r['tb'], 2500)
    ys = r['sol'].sol(ts)
    ax.semilogx(-(ts - t_c), np.exp(ys[4]), label=f'$\\rho_c = {rho_c}$')
ax.set_xlabel('|t - t_c| before bounce/crunch'); ax.set_ylabel('a')
ax.set_title('Contraction: both models'); ax.legend(fontsize=8)

ax = axs[1]
ax.plot(tgr_s, agr, 'k-', lw=2, label='GR')
for rho_c in RHO_C_LIST:
    r = eff_runs[rho_c]
    ts = np.linspace(r['tb'] - 4.0, r['sol'].t[-1], 2500)
    ys = r['sol'].sol(ts)
    ax.plot(ts - t_c, np.exp(ys[4]), label=f'$\\rho_c = {rho_c}$')
ax.axvline(0, color='grey', ls=':', lw=1)
ax.set_xlim(-0.5, 0.1)
ax.set_ylim(0.15, 0.65)
ax.set_xlabel('t - t_c'); ax.set_ylabel('a')
ax.set_title('Near the bounce: GR crunches, effective model bounces')
ax.legend(fontsize=8)

ax = axs[2]
r = r041
ts = np.linspace(r['tb'] - 2.0, r['sol'].t[-1], 2500)
ys = r['sol'].sol(ts)
rho_ts = ek.rho_tot(r['pot'], ys[0], ys[1], ys[3])
ax.plot(ts - t_c, rho_ts / 0.41, label=r'$\rho/\rho_c$')
ax.plot(ts - t_c, ys[2] / np.sqrt(0.41 / 3), label=r'$H/\sqrt{\rho_c/3}$')
ax.axvline(0, color='grey', ls=':', lw=1)
ax.set_xlabel('t - t_c')
ax.set_title('Bounce window, rho_c = 0.41')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGS / 'fig3_bounce.png')
plt.show()

metrics['steps']['3'] = {
    'p': p, 'rho_c_list': bounce_table,
    'smoothing_factor_to_rho_0.41': smooth_factor,
    'n_efolds_to_rho_0.41': float(n_sm),
    't_c_offset': t_c,
    'post_bounce_expansion_factor': exp_factor,
    'notes': ('effective Friedmann equation specified by hand; '
              'model recollapses after the bounce (rho -> 0 in expansion); '
              'crunch time t is shifted by the marginal time-translation '
              'mode, t-invariant observables are unaffected'),
}


rho_c=0.041   t_bounce=   -4.1295  a_bounce=0.2513  rho(t_bounce)=0.041000  a_GR(rho=rho_c)=0.2473


rho_c=0.41    t_bounce=   -3.8197  a_bounce=0.2240  rho(t_bounce)=0.410000  a_GR(rho=rho_c)=0.2204


rho_c=4.1     t_bounce=   -3.7217  a_bounce=0.1996  rho(t_bounce)=4.100000  a_GR(rho=rho_c)=0.1964

By rho = 0.41 the contraction lasted 1.51 e-folds and the anisotropy ratio fell by a factor 6.38e-10 (p=0.1).
GR crunch time offset t_c = -3.8724 (time-translation mode; linear in the initial shear fraction)
Post-bounce expansion before recollapse: a_max/a_bounce = 1.0115


## Step 4 — WKB classicality of the ekpyrotic phase

Minisuperspace Wheeler–DeWitt equation for $(\alpha = \ln a,\ \phi)$,
with the factor ordering that removes first derivatives and $\hbar$ kept
explicit:

$$\Big[\hbar^2\Big(\frac{\partial^2}{\partial\alpha^2}
- 6\frac{\partial^2}{\partial\phi^2}\Big)
+ 12\,a^6 V(\phi)\Big]\Psi = 0 .$$

A WKB wavefunction $\Psi = A\,e^{iS/\hbar}$ is *classical* when the
amplitude/transport equation is dominated by the phase, i.e. when

$$\mathcal Q \;\equiv\;
\frac{\hbar\,|\nabla^2 S|}{|(\nabla S)^2|} \;\ll\; 1,
\qquad
(\nabla S)^2 = -(S_{,\alpha})^2 + 6(S_{,\phi})^2 = -12 a^6 V .$$

Along the classical trajectory $S_{,\alpha} = -6 a^3 H$,
$S_{,\phi} = a^3 \dot\phi$, so $\mathcal Q$ is computable from the
numerical solution without further approximation. On the attractor one finds
analytically

$$\mathcal Q \;=\; \frac{\hbar}{2p}\frac{|t|}{a^3}
\;\propto\; a^{(1-3p)/p} \;\xrightarrow[a \to 0]{}\; 0
\quad (p<1/3),$$

i.e. **the WKB description of the contracting branch improves monotonically
as the ekpyrotic phase proceeds** — the behaviour underlying the
classicality analyses of the ekpyrotic no-boundary/tunnelling wavefunctions
(Battarra & Lehners 2014; Lehners 2015). We verify the exponent numerically.

**Caveats (kept for step 5):** this is a self-consistency criterion for a
WKB state peaked on the classical trajectory — it says nothing by itself
about the boundary proposal that selects the saddle; the factor ordering of
the WDW equation is a choice; $\mathcal Q$ scales linearly with $\hbar$,
and we quote $\hbar = 1$; minisuperspace removes the inhomogeneous modes
whose quantum fluctuations are the actual observables.


In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))

Q_rows = []
for p in P_LIST:
    sol, pot = ek.integrate_gr(p, V0, t_i, SHEAR_FRAC, rho_stop=RHO_STOP)
    n = np.linspace(0, sol.t[-1], 1500)
    y = sol.sol(n)
    phi, u, h, R, t = y
    a = np.exp(-n)
    Q = ek.classicality_Q(pot, -n, phi, u, -h, hbar=1.0)
    rho_p = 0.5 * u**2 + pot[0](phi)
    win = (R / rho_p) < 1e-3  # same perturbative window as step 1
    if win.sum() > 30:
        m_q, r2_q = ek.fit_exponent(a[win], Q[win])
    else:
        m_q, r2_q = ek.fit_exponent(a, Q)
    Q_rows.append({'p': p, 'Q_exp_fit': float(m_q),
                   'Q_exp_theory': ek.classicality_exponent(p),
                   'Q_R2': float(r2_q), 'Q_i': float(Q[0]),
                   'Q_final': float(Q[-1])})
    ax1.loglog(a, Q, label=f'p={p}')

p_ref = 0.1
sol, pot = ek.integrate_gr(p_ref, V0, t_i, SHEAR_FRAC, rho_stop=RHO_STOP)
n = np.linspace(0, sol.t[-1], 1500)
y = sol.sol(n)
a_ref = np.exp(-n)
Q_ref = ek.classicality_Q(pot, -n, y[0], y[1], -y[2], hbar=1.0)
ax1.loglog(a_ref, Q_ref[-1] * (a_ref / a_ref[-1]) ** ek.classicality_exponent(p_ref),
           'k--', lw=1, label=r'$\propto a^{(1-3p)/p}$')
ax1.axhline(1.0, color='grey', ls=':', lw=1)
ax1.invert_xaxis()
ax1.set_xlabel('scale factor a (contraction: decreasing)')
ax1.set_ylabel('classicality Q (hbar = 1)')
ax1.set_title('WKB classicality quantity along contraction')
ax1.legend(fontsize=8)

# normalized view: Q/Q_i vs a for the p=0.1 run at three hbar values
for hb in (1.0, 1e-3, 1e-6):
    ax2.loglog(a_ref, hb * Q_ref / Q_ref[0], label=f'$\\hbar = {hb:g}$')
ax2.invert_xaxis()
ax2.set_xlabel('scale factor a (contraction: decreasing)')
ax2.set_ylabel('Q / Q_i (p = 0.1)')
ax2.set_title('Self-improvement of the WKB description')
ax2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / 'fig4_classicality.png')
plt.show()

print('p      Q exponent fit / theory          R^2        Q(t_i)      Q(cutoff)')
for r in Q_rows:
    print(f"{r['p']:<6} {r['Q_exp_fit']:>9.4f} / {r['Q_exp_theory']:<9.4f}   "
          f"{r['Q_R2']:.8f}   {r['Q_i']:<10.3e} {r['Q_final']:.3e}")

# scale factor at which Q crosses 1 (hbar = 1), for p = 0.1
q_exp = ek.classicality_exponent(p_ref)
a_q1 = float((1.0 / Q_ref[0]) ** (1.0 / q_exp))
print(f'\np=0.1, hbar=1: Q crosses 1 at a/a_i ~ {a_q1:.3e}')
metrics['steps']['4'] = {
    'Q_rows': Q_rows,
    'a_at_Q_equal_1_p0.1_hbar1': a_q1,
    'notes': ('self-consistency criterion for a WKB state on the classical '
              'trajectory; hbar kept as a free overall factor'),
}


p      Q exponent fit / theory          R^2        Q(t_i)      Q(cutoff)
0.01     97.0000 / 97.0000     1.00000000   5.000e+07  2.091e-05
0.05     17.0000 / 17.0000     1.00000000   1.000e+07  5.583e-04
0.1       7.0000 / 7.0000      1.00000000   5.000e+06  2.924e-02
0.3       0.3334 / 0.3333      0.99999998   1.668e+06  1.240e+05
0.5      -0.9972 / -1.0000     0.99999922   9.997e+05  5.064e+02

p=0.1, hbar=1: Q crosses 1 at a/a_i ~ 1.104e-01


## Step 5 — What the models predict, assume, and leave unverified

Condensed here; the full discussion is in
[results/OD-QC-001_REPORT.md](../results/OD-QC-001_REPORT.md).

**Reproduced (numerically verified against analytic scalings):**

* $\rho_\phi \propto a^{-3(1+w)}$ and
  $\rho_\Sigma/\rho_\phi \propto a^{3(w-1)}$ along the ekpyrotic
  attractor, for all $w>1$ cases tested;
* the $w<1$ counterexample shows the ratio growing during contraction:
  $w>1$ is the operative condition, not contraction per se;
* the smoothing exponent is insensitive to the shear amplitude and to $V_0$;
  the final ratio is linear in the initial shear fraction;
* the WKB quantity $\mathcal Q \propto a^{(1-3p)/p}$: the semiclassical
  (WKB) description of the contracting branch improves monotonically as the
  crunch is approached, for $p<1/3$.

**Assumed without derivation here:** flat slicing; single canonical scalar;
exact exponential potential; shear as an adiabatic stiff component
(= Bianchi I); the effective Friedmann equation of step 3, which is
specified, not derived; the WDW factor ordering of step 4; $\hbar$ as a
free overall factor in $\mathcal Q$.

**Shown to be missing / not delivered by the simplified models:**

* No bounce exists in the GR model (classical crunch at finite time).
* The specified effective model bounces at $\rho = \rho_c$ but (with
  $V<0$) recollapses afterwards — connecting to a hot expanding phase needs
  extra physics (e.g. the ghost-condensate/galileon constructions of the
  ekpyrotic literature, reheating).
* Nothing here addresses perturbations across the bounce, the quantum
  boundary proposal (no-boundary vs tunnelling), or anisotropies
  inhomogeneously coupled — the items a full classicality claim requires.


## Step 6 — Follow-ups: attractor stability, the bounce cascade, and a failed remedy

Three questions left open by steps 3–5, now answered quantitatively.

**(a) Attractor stability.** Writing the pure-scalar dynamics in scaled
variables $X=-c(\phi-\phi_{\rm att})$, $U=u/u_{\rm att}$,
$H_h=h/h_{\rm att}$ (all $t$-invariant relative coordinates, fixed point
$(X,U,H_h)=(0,1,1)$), the flow reduces exactly to the autonomous system

$$\dot X=+\tfrac{2}{p}(U/H_h-1),\quad
  \dot U=(3-\tfrac1p)U+\tfrac{1-3p}{p}\frac{e^{X}}{H_h},\quad
  \dot H_h=\frac{U^2/H_h-H_h}{p},$$

whose linearisation has the exact spectrum

$$\lambda_0=0,\qquad
  \lambda_1=-\frac{1-3p}{p}=-\frac{3}{2}(w-1),\qquad
  \lambda_2=-\frac{2}{p}=-3(w+1),$$

(all real). $\lambda_0=0$ is the time-translation mode of step 5
(eigenvector $(2,1,1)$: a pure shift of the crunch time). The slow mode
decays as $a^{+3(w-1)/2}$ — exactly half the smoothing exponent — and the
fast mode as $a^{3(w+1)}$, so generic homogeneous perturbations die within
a fraction of an e-fold for $w\gg1$ (e.g. $e^{-7n}$ for $p=0.1$, measured
$6.99$). Crucially $\lambda_1=0 \iff w=1$: **the $w=1$ threshold is not
only the smoothing condition but the exact linear stability boundary of the
scaling solution** — for $w<1$ scalar deviations grow as
$a^{3(w-1)/2}\to\infty$ during contraction.

**(b) The bounce cascade.** Continuing the effective model past its first
recollapse, the cycles follow $\Delta t_n = A/n^{2}$ ($\gamma=1.998$ fit),
with $|u|_n\simeq 5.2\,n$ and the field gliding down the potential: an
infinite sequence of bounces with unbounded kinetic energy accumulating at a
finite time — a Zeno-type breakdown of the effective closure, not physics.

**(c) A failed remedy.** Flipping the sign of $V$ in the expanding branch
(a crude stand-in for "the potential turns positive") does not help: the
post-bounce velocity $u_b=-2.4$ points toward larger $|V|$, and the
solution is driven back to the constraint boundary in finite time. The
bounce mechanism must alter the field's post-bounce roll (as ghost
condensate / galileon constructions do), not merely deform $H^2$.


In [6]:
# ---- (a) eigenvalues of the scaled system across p, vs exact formulas ----
def scaled_eigenvalues(p):
    K = (1 - 3*p) / p
    def F(Y):
        X, U, Hh = Y
        return np.array([
            (2/p)*(U/Hh - 1),
            (3 - 1/p)*U + K*np.exp(X)/Hh,
            (U**2/Hh - Hh)/p,
        ])
    Y0 = np.array([0.0, 1.0, 1.0])
    J = np.zeros((3, 3))
    for j in range(3):
        e = np.zeros(3); e[j] = 1e-7
        J[:, j] = (F(Y0 + e) - F(Y0 - e)) / 2e-7
    return np.sort(np.linalg.eigvals(J).real)

pp = np.linspace(0.03, 0.6, 300)
lam = np.array([scaled_eigenvalues(x) for x in pp])
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(pp, lam[:, 0], 'b-', lw=1.5, label=r'$\lambda_1$ (slow mode)')
ax.plot(pp, lam[:, 1], 'c:', lw=1.5, label=r'$\lambda_2$ (fast mode)')
ax.plot(pp, lam[:, 2], 'k--', lw=1, label=r'$\lambda_0$ (marginal)')
ax.plot(pp, -(1 - 3*pp)/pp, 'r--', lw=1, label=r'$-(1-3p)/p$ (exact)')
ax.plot(pp, -2/pp, 'g--', lw=1, label=r'$-2/p$ (exact)')
ax.axvline(1/3, color='grey', ls=':', lw=1)
ax.axhline(0, color='grey', lw=0.5)
ax.set_xlabel('p'); ax.set_ylabel('eigenvalue (per e-fold)')
ax.set_title('Stability of the ekpyrotic scaling solution: $\\lambda_1 = 0$ at $w=1$')
ax.set_ylim(-25, 3)
ax.legend(fontsize=8, loc='lower left')
fig.tight_layout()
fig.savefig(FIGS / 'fig5_stability.png')
plt.show()

print('numeric vs exact (p=0.1):', scaled_eigenvalues(0.1), ' exact: {0, -7, -20}')
print('numeric vs exact (p=0.3):', scaled_eigenvalues(0.3), ' exact: {0, -1/3, -20/3}')
print('at p = 1/3 + eps:', scaled_eigenvalues(1/3 + 1e-9))
metrics['steps']['6a'] = {
    'spectrum': 'lambda = {0, -(1-3p)/p = -3(w-1)/2, -2/p = -3(w+1)}',
    'stability_boundary_p': 1/3,
    'numeric_p0.1': [float(x) for x in scaled_eigenvalues(0.1)],
}

# ---- convergence of a perturbed full run onto a shifted attractor ----
p, V0, t_i = 0.1, 1.0, -1e6
c = np.sqrt(2/p)
sol0, pot0 = ek.integrate_gr(p, V0, t_i, 1e-4)
n_end = sol0.t[-1]
n = np.linspace(0, n_end, 600)
ya = sol0.sol(n)
y0 = ek.initial_state(p, V0, t_i, 1e-4, du_rel=1e-2)
sol, _ = ek.integrate_gr(p, V0, t_i, 1e-4, y0=y0)
y = sol.sol(n)
# scaled coordinates along both runs
att = ek.attractor_n(p, V0, n, t_i)
Xa = -c*(ya[0] - att['phi']);  Xp = -c*(y[0] - att['phi'])
Ua = ya[1]/att['u'];           Up = y[1]/att['u']
Ha = -ya[2]/att['h'];          Hp = -y[2]/att['h']
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.semilogy(n, np.abs(Xp - Xa), label=r'$|X|$')
ax.semilogy(n, np.abs(Up - Ua), label=r'$|U-1|$')
ax.semilogy(n, np.abs(Hp - Ha), label=r'$|H_h-1|$')
ax.set_xlabel('e-folds n'); ax.set_ylabel('deviation from attractor slice')
ax.set_title('Off-attractor run (du/u = 1%): transient + marginal offset')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / 'fig6_convergence.png')
plt.show()
# each component: decaying transient + constant marginal offset; fit the
# transient envelope of |U-1 - (U-1)_final|
for nm, d in (('X', Xp - Xa), ('U', Up - Ua), ('Hh', Hp - Ha)):
    res = np.abs(d - d[-1])
    win = (res > 1e-13) & (n < 1.5)
    m, r2 = ek.fit_exponent(np.exp(-n[win]), res[win])
    print(f'{nm:3s}: transient decay rate = {m:8.3f} / e-fold (theory 7.00), '
          f'marginal offset = {d[-1]:+.3e}')
metrics['steps']['6a'].update({'measured_decay_p0.1_du1e-2':
    'see printed rates; theory (1-3p)/p = 7.00 for p = 0.1'})


numeric vs exact (p=0.1): [-2.00000000e+01 -6.99999999e+00 -5.55111690e-09]  exact: {0, -7, -20}
numeric vs exact (p=0.3): [-6.66666667e+00 -3.33333332e-01 -1.85036719e-09]  exact: {0, -1/3, -20/3}
at p = 1/3 + eps: [-5.99999998e+00 -1.43616119e-09  1.04361610e-08]
X  : transient decay rate =    6.990 / e-fold (theory 7.00), marginal offset = -6.558e-02
U  : transient decay rate =    6.997 / e-fold (theory 7.00), marginal offset = -3.226e-02
Hh : transient decay rate =    6.998 / e-fold (theory 7.00), marginal offset = +3.226e-02


In [7]:
# ---- (b) the bounce cascade ----
p, V0, t_i = 0.1, 1.0, -1e6
sc, pote = ek.integrate_effective(p, V0, t_i, 1e-4, rho_c=0.41,
                                  stop_at_rho0=False, floor_factor=1e-8,
                                  t_end=1e6)
tb = sc.t_events[0]
iv = np.diff(tb)
ncy = len(iv)
print(f'{len(tb)} H-crossings between t={tb[0]:.4f} and t={tb[-1]:.5f} '
      f'({ncy} cycles, resolved down to rho = 1e-8 rho_c)')

from scipy.optimize import curve_fit
nn = np.arange(1, ncy + 1)
powl = lambda k, A, g: A / k**g
coef, _ = curve_fit(powl, nn[8:], iv[8:], p0=[0.1, 2.0], maxfev=20000)
A, g = coef
resid = np.max(np.abs(iv[8:] - powl(nn[8:], A, g)) / iv[8:])
print(f'interval law: dt_n = {A:.4f} / n^{g:.4f}   (max rel residual {resid:.1e})')
uu = np.array([abs(sc.sol(t)[1]) for t in tb[1:]])   # velocity at cycle ends
nn = np.arange(1, ncy + 1)
sl, ic = np.polyfit(nn, uu, 1)[0], np.polyfit(nn, uu, 1)[1]
print(f'field velocity: |u|_n = {sl:.2f} n + {ic:.2f}  -> diverges at the accumulation')
t_acc = tb[-1] + (A / (g - 1)) * ncy**(1 - g)
print(f'accumulation time t_acc ~ {t_acc:.6f} (tail sum of 1/n^2 series)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.loglog(nn, iv, 'k.', ms=3)
ax1.loglog(nn, A / nn**g, 'r-', lw=1,
           label=f'$0.086/n^{{2}}$ (fit g={g:.3f})')
ax1.set_xlabel('cycle number n'); ax1.set_ylabel('cycle duration $\\Delta t_n$')
ax1.set_title('Bounce cascade: cycle duration'); ax1.legend(fontsize=8)
ax2.plot(nn, uu, 'k.', ms=3)
ax2.plot(nn, sl*nn + ic, 'r-', lw=1, label=f'|u| = {sl:.1f} n')
ax2.set_xlabel('cycle number n'); ax2.set_ylabel('|field velocity| at crossing')
ax2.set_title('Kinetic energy diverges'); ax2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / 'fig7_cascade.png')
plt.show()
metrics['steps']['6b'] = {'n_cycles_resolved': int(ncy),
                          'interval_law_A': float(A), 'interval_law_gamma': float(g),
                          'u_growth_per_cycle': float(sl),
                          't_acc_estimate': float(t_acc),
                          'note': 'Zeno-type accumulation: infinite bounces, '
                                  'divergent |u|, finite time'}


116 H-crossings between t=-3.8197 and t=-3.62613 (115 cycles, resolved down to rho = 1e-8 rho_c)
interval law: dt_n = 0.0905 / n^2.0136   (max rel residual 1.5e-02)
field velocity: |u|_n = 5.19 n + 2.17  -> diverges at the accumulation
accumulation time t_acc ~ -3.625399 (tail sum of 1/n^2 series)


In [8]:
# ---- (c) failed remedy: flip the potential sign in the expanding branch ----
from scipy.integrate import solve_ivp
p, V0, t_i = 0.1, 1.0, -1e6
c = np.sqrt(2/p); rho_c = 0.41
s0, pote = ek.integrate_effective(p, V0, t_i, 1e-4, rho_c=rho_c)
tb0 = s0.t_events[0][0]
yb = s0.sol(tb0)
print(f'bounce state: u_b = {yb[1]:.3f} (directed toward larger |V|), '
      f'phi_b = {yb[0]:.3f}')

# on-shell check of the expanding branch with V > 0: evolve (phi, u, R, alpha)
# with H fixed by the constraint; monitor rho
def rhs_onshell(t, y):
    phi, u, R, alpha = y
    V = V0*np.exp(-c*phi)
    rho = 0.5*u*u + V + R
    H = np.sqrt(max(rho*(1 - rho/rho_c), 0.0)/3.0)
    return [u, -3*H*u - c*V, -6*H*R, H]

def ev_break(t, y):
    return 0.5*y[1]**2 + V0*np.exp(-c*y[0]) + y[3] - 1.01*rho_c
ev_break.terminal = True; ev_break.direction = 1

s2 = solve_ivp(rhs_onshell, (tb0, tb0 + 5.0),
               [yb[0], yb[1], yb[3], yb[4]], method='DOP853',
               rtol=1e-10, atol=1e-13, events=[ev_break])
rho_t = 0.5*s2.y[1]**2 + V0*np.exp(-c*s2.y[0]) + s2.y[2]
print(f'V>0 expanding branch: integration terminates {s2.t[-1] - tb0:+.4f} '
      f'after the bounce (status {s2.status}: step failure / overflow)')
print(f'  rho leaves the allowed band [0, rho_c] within that time '
      f'(final rho ~ {rho_t[-1]:.1e}, |u| ~ {abs(s2.y[1,-1]):.1e});')
print('  the field roll (u_b < 0, V_,phi > 0) drives the state back across '
      'the constraint boundary:\n  the effective model has no valid '
      'expanding branch without new physics.')
metrics['steps']['6c'] = {
    'u_bounce': float(yb[1]),
    'breakdown_dt': float(s2.t[-1] - tb0),
    'note': 'sign flip does not rescue expansion; rho returns to the '
            'constraint boundary in finite time',
}


bounce state: u_b = -2.435 (directed toward larger |V|), phi_b = -0.210
V>0 expanding branch: integration terminates +0.1929 after the bounce (status -1: step failure / overflow)
  rho leaves the allowed band [0, rho_c] within that time (final rho ~ 1.9e+26, |u| ~ 1.4e+13);
  the field roll (u_b < 0, V_,phi > 0) drives the state back across the constraint boundary:
  the effective model has no valid expanding branch without new physics.


## Step 7 — The analytic origin of the $\gamma=2$ cascade law

The empirical law of step 6, $\Delta t_n = A/n^2$, is derivable in three
short steps. Work at fixed $\rho_c$, asymptotically in the cycle number $n$
(so $|V| \gg \rho_c$ at the turnarounds), and neglect $R$.

**(i) Cycle time.** Along the effective flow, $\rho+p = u^2$ and
$H=\sqrt{\rho(1-\rho/\rho_c)/3}$, so the time for one traversal
$\rho_c\to0$ or $0\to\rho_c$ at approximately constant $u$ is

$$T_{1/2}=\int_0^{\rho_c}\frac{\dd\rho}{3Hu^2}
  =\frac{1}{\sqrt3\,u^2}\int_0^{\rho_c}\frac{\dd\rho}
     {\sqrt{\rho(1-\rho/\rho_c)}}
  =\frac{\pi\sqrt{\rho_c/3}}{u^2},$$

the $\rho$-integral being elementary ($\rho=\rho_c\sin^2\theta$). A full
bounce-to-bounce cycle has $T(u)=2\pi\sqrt{\rho_c/3}/u^2$.

**(ii) Velocity growth.** Over a cycle, $\Delta u=\oint\dot u\,\dd t$
with $\dot u=-3Hu-c|V|$. The friction term integrates to
$-3\oint H\,\dd t=-3\,\Delta\ln a\approx0$ ($a$ returns to its
bounce value), and $|V|$ is near-constant across a cycle (the field moves
by only $\Delta\phi\sim1/u$), so

$$\Delta u \simeq -c\,|V|\,T
  = -c\,\frac{u^2}{2}\cdot\frac{2\pi\sqrt{\rho_c/3}}{u^2}
  = -c\pi\sqrt{\rho_c/3} \;\equiv\; -\alpha,$$

using the turnaround constraint $|V|=u^2/2-\rho_c+R\simeq u^2/2$: the
$u^2$ **cancels** and the growth per cycle is a constant,
$\alpha=c\pi\sqrt{\rho_c/3}$. Hence $u_n\simeq\alpha n$.

**(iii) The law.** Substituting,

$$\Delta t_n=\frac{2\pi\sqrt{\rho_c/3}}{u_n^2}
  =\frac{2\pi\sqrt{\rho_c/3}}{\alpha^2}\,\frac{1}{n^2}
  =\frac{3p}{\pi\sqrt{3\rho_c}}\;\frac{1}{n^2},$$

i.e. $\gamma=2$ **exactly**, with $A=3p/\pi\sqrt{3\rho_c}$. Two further
predictions follow with no free parameters:

* **Field drift:** $\Delta\phi_n=\oint u\,\dd t\simeq-u_nT_n
  =-2/(cn)$, so $\phi_n=\phi_0-(2/c)\ln n$ — a *harmonic*, not
  geometric, creep (the apparent asymptote of step 6 is $\ln n$).
* **Accumulation time:** $t_{\rm acc}-t_N
  =A\sum_{k>N}k^{-2}=A\,(\pi^2/6-H_N^{(2)})\simeq A/N$.

All four are confirmed below, for two values of $p$ (which also tests the
predicted $A\propto p$ scaling): $\alpha$ to four significant digits, the
scaled cycle time $\Delta t_nu_n^2/2\pi\sqrt{\rho_c/3}=1$ to
$\sim2\%$ (the residual being the $1/n$ transient at small $n$), the
$\phi$-law to $0.4\%$, and $t_{\rm acc}$ to the digit.


In [9]:
# ---- step 7: verification of the analytic cascade law ----
from scipy.optimize import curve_fit

def cascade_check(p, rho_c=0.41, V0=1.0, t_i=-1e6):
    c = np.sqrt(2/p)
    alpha_pred = c*np.pi*np.sqrt(rho_c/3)
    A_pred = 3*p/(np.pi*np.sqrt(3*rho_c))
    sc, _ = ek.integrate_effective(p, V0, t_i, 1e-4, rho_c=rho_c,
                                   stop_at_rho0=False, floor_factor=1e-8,
                                   t_end=1e6)
    tb = sc.t_events[0]                      # bounces: full cycles
    iv = np.diff(tb); n = np.arange(1, len(iv)+1)
    uu = np.array([abs(sc.sol(t)[1]) for t in tb[1:]])
    alpha_m = np.polyfit(n, uu, 1)[0]
    ratio = iv * uu**2 / (2*np.pi*np.sqrt(rho_c/3))
    ph = np.array([sc.sol(t)[0] for t in tb[1:]])
    dphi_ratio = np.mean((ph[1:]-ph[:-1]) / (-2/(c*n[1:])))
    t_acc = tb[-1] + A_pred*(np.pi**2/6 - np.sum(n**-2.0))
    return dict(p=p, alpha_pred=alpha_pred, alpha_m=alpha_m,
                A_pred=A_pred, ratio_mean=ratio[8:].mean(),
                ratio_std=ratio[8:].std(), dphi_ratio=dphi_ratio,
                t_acc=t_acc, n_cycles=len(iv), tb=tb, iv=iv, n=n, uu=uu)

checks = [cascade_check(0.1), cascade_check(0.05)]
print('p     alpha (pred)   alpha (meas)   dt*u^2/2pi..    dphi ratio   A_pred   n_cyc')
for r in checks:
    print(f"{r['p']:<5} {r['alpha_pred']:<14.4f} {r['alpha_m']:<14.4f} "
          f"{r['ratio_mean']:.4f}+-{r['ratio_std']:.1e}   {r['dphi_ratio']:.4f}       "
          f"{r['A_pred']:.5f}  {r['n_cycles']}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
for r, col in zip(checks, ('C0', 'C1')):
    nn, iv = r['n'], r['iv']
    ax1.loglog(nn, iv*nn**2, '.', ms=3, color=col,
               label=f"p={r['p']}: $A_n=\\Delta t_n n^2$ (pred {r['A_pred']:.4f})")
    ax1.axhline(r['A_pred'], color=col, ls='--', lw=1)
    ax2.plot(nn, r['uu']/r['alpha_pred'], '.', ms=3, color=col,
             label=f"p={r['p']}: $u_n/\\alpha$ (slope 1)")
    ax2.plot(nn, nn, '-', lw=1, color=col)
ax1.set_xlabel('cycle n'); ax1.set_ylabel(r'$\Delta t_n\, n^2$')
ax1.set_title(r'$\Delta t_n\, n^2 = A = 3p/\pi\sqrt{3\rho_c}$')
ax1.legend(fontsize=8)
ax2.set_xlabel('cycle n'); ax2.set_ylabel('$u_n/\\alpha$')
ax2.set_title('Velocity growth: $u_n = \\alpha n$, $\\alpha=c\\pi\\sqrt{\\rho_c/3}$')
ax2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / 'fig8_cascade_law.png')
plt.show()

metrics['steps']['7'] = {
    'law': 'dt_n = A/n^2, A = 3p/(pi*sqrt(3*rho_c)); u_n = alpha*n, '
           'alpha = c*pi*sqrt(rho_c/3); dphi_n = -2/(c n); '
           't_acc - t_N = A (pi^2/6 - H_N^(2))',
    'checks': [{k: v for k, v in r.items()
                if k in ('p', 'alpha_pred', 'alpha_m', 'A_pred',
                         'ratio_mean', 'ratio_std', 'dphi_ratio', 'n_cycles')}
               for r in checks],
}


p     alpha (pred)   alpha (meas)   dt*u^2/2pi..    dphi ratio   A_pred   n_cyc
0.1   5.1939         5.1938         1.0250+-2.2e-02   1.0037       0.08610  115
0.05  7.3453         7.3452         1.0261+-2.2e-02   1.0021       0.04305  108


In [10]:
out = {'parameters': {'V0': V0, 't_i': t_i, 'shear_frac': SHEAR_FRAC,
                      'rho_stop': RHO_STOP, 'p_list': P_LIST},
       **metrics}
with open(ROOT / 'results' / 'metrics.json', 'w') as f:
    json.dump(out, f, indent=2)
print('metrics written:', ROOT / 'results' / 'metrics.json')


metrics written: /workspace/nucleus/work/projects/p-afe332a105bf4aed822a5d284f83970c/repo/results/metrics.json
